# GenomeQuery on Google Colab
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR_GITHUB_USER/YOUR_REPO/blob/main/notebooks/genomequery_colab.ipynb)

Nothing to install. The first cells write the engine package into the Colab runtime, then the notebook walks through search, alignment, the workflow engine, the relational store and benchmarks. Choose **Runtime → Run all**.
Contact: harshinisuresha7@gmail.com

In [ ]:
!mkdir -p engine

In [ ]:
%%writefile engine/__init__.py


In [ ]:
%%writefile engine/seqtools.py
"""Sequence utilities, k-mer index, trie and reference scan used by the GenomeQuery workbench."""
import random, time, tracemalloc
from collections import Counter

def clean(s):
    return ''.join(c for c in s.upper() if c.isalpha())

def kind_of(s):
    return 'DNA' if set(s) <= set('ACGTN') else 'PROTEIN'

def gc(s):
    return (s.count('G') + s.count('C')) / len(s) if s else 0.0

def parse_fasta(text):
    out, name, buf = {}, None, []
    def flush():
        if name is not None and buf:
            key, i = name, 2
            while key in out: key, i = f"{name}_{i}", i + 1
            out[key] = clean(''.join(buf))
    for line in text.splitlines():
        line = line.strip()
        if not line: continue
        if line.startswith('>'):
            flush(); name = (line[1:].split() or [f'seq{len(out)+1}'])[0]; buf = []
        else:
            if name is None: name = 'seq1'
            buf.append(line)
    flush()
    return {k: v for k, v in out.items() if v}

def mutate(s, rate, rng, indel=0.01):
    out = []
    for c in s:
        r = rng.random()
        if r < indel / 2: continue
        if r < indel: out.append(c); out.append(rng.choice('ACGT')); continue
        out.append(rng.choice([x for x in 'ACGT' if x != c]) if r < indel + rate else c)
    return ''.join(out)

def synthetic(n=12, length=2000, gc_target=.5, motif='GATTACAGATTACA', related=.4, rate=.6, seed=1):
    rng = random.Random(seed); w = [(1-gc_target)/2, gc_target/2, gc_target/2, (1-gc_target)/2]
    base = ''.join(rng.choices('ACGT', w, k=length)); out = {}
    for i in range(n):
        s = base if i == 0 else (mutate(base, .08, rng) if rng.random() < related else ''.join(rng.choices('ACGT', w, k=length)))
        if motif and rng.random() < rate and len(s) > len(motif) + 1:
            p = rng.randrange(0, len(s) - len(motif)); s = s[:p] + motif + s[p+len(motif):]
        out[f'seq_{i+1:03d}'] = s
    return out

def naive_find(seq, q):
    out, i = [], seq.find(q)
    while i != -1: out.append(i); i = seq.find(q, i + 1)
    return out

def hamming(a, b):
    return sum(x != y for x, y in zip(a, b))

class KmerIndex:
    """Hash table from every k-mer to (sequence, position) pairs."""
    def __init__(self, k):
        self.k, self.table, self.names, self.seqs = k, {}, [], []
    def add(self, name, seq):
        sid = len(self.names); self.names.append(name); self.seqs.append(seq); k, t = self.k, self.table
        for i in range(len(seq) - k + 1):
            kmer = seq[i:i+k]
            if kmer in t: t[kmer].append((sid, i))
            else: t[kmer] = [(sid, i)]
    def exact(self, q):
        if len(q) < self.k: return []
        return [(self.names[s], p) for s, p in self.table.get(q[:self.k], ()) if self.seqs[s].startswith(q, p)]
    def approx(self, q, d):
        """Seed-and-vote: every query k-mer votes for a start position; candidates are verified by Hamming distance."""
        k, votes = self.k, Counter()
        for j in range(len(q) - k + 1):
            for s, p in self.table.get(q[j:j+k], ()):
                votes[(s, p - j)] += 1
        rows = []
        for (s, st), v in votes.items():
            seq = self.seqs[s]
            if st < 0 or st + len(q) > len(seq): continue
            mm = hamming(q, seq[st:st+len(q)])
            if mm <= d: rows.append((self.names[s], st, mm, v))
        return sorted(rows, key=lambda r: (r[2], r[0], r[1]))
    def guaranteed(self, qlen, d):
        """With d mismatches at most d*k k-mers are broken, so one survives if qlen-k+1 > d*k."""
        return qlen - self.k + 1 > d * self.k
    def stats(self):
        n = sum(len(v) for v in self.table.values())
        return dict(distinct=len(self.table), entries=n, avg_bucket=n / max(1, len(self.table)), max_bucket=max((len(v) for v in self.table.values()), default=0))
    def bucket_sizes(self):
        return Counter(len(v) for v in self.table.values())

def build_index(seqs, k):
    tracemalloc.start(); t = time.perf_counter(); idx = KmerIndex(k)
    for n, s in seqs.items(): idx.add(n, s)
    secs = time.perf_counter() - t; _, peak = tracemalloc.get_traced_memory(); tracemalloc.stop()
    return idx, secs, peak / 1e6

class Trie:
    def __init__(self): self.root, self.nodes, self.words = {}, 1, 0
    def insert(self, word, payload):
        node = self.root
        for ch in word:
            nxt = node.get(ch)
            if nxt is None: nxt = node[ch] = {}; self.nodes += 1
            node = nxt
        node.setdefault('$', []).append(payload); self.words += 1
    def prefix(self, p, limit=500):
        node = self.root
        for ch in p:
            node = node.get(ch)
            if node is None: return []
        out, stack = [], [node]
        while stack and len(out) < limit:
            n = stack.pop()
            if '$' in n: out.extend(n['$'])
            stack.extend(v for k, v in n.items() if k != '$')
        return out[:limit]


In [ ]:
%%writefile engine/align.py
"""Dynamic-programming alignment: Smith-Waterman (local) and Needleman-Wunsch (global), linear gap penalty."""
import numpy as np

def align(a, b, mode='local', match=2, mismatch=-1, gap=-2):
    n, m, loc = len(a), len(b), mode == 'local'
    H = [[0] * (m + 1) for _ in range(n + 1)]; T = [[0] * (m + 1) for _ in range(n + 1)]
    if not loc:
        for i in range(1, n + 1): H[i][0] = i * gap; T[i][0] = 2
        for j in range(1, m + 1): H[0][j] = j * gap; T[0][j] = 3
    best, bi, bj = 0, 0, 0
    for i in range(1, n + 1):
        ai, Hi, Hp, Ti = a[i-1], H[i], H[i-1], T[i]
        for j in range(1, m + 1):
            d = Hp[j-1] + (match if ai == b[j-1] else mismatch); u = Hp[j] + gap; l = Hi[j-1] + gap
            v = max(d, u, l); t = 1 if v == d else 2 if v == u else 3
            if loc and v <= 0: v, t = 0, 0
            Hi[j] = v; Ti[j] = t
            if loc and v > best: best, bi, bj = v, i, j
    if not loc: best, bi, bj = H[n][m], n, m
    i, j, ra, rb, rm, path = bi, bj, [], [], [], []
    while i > 0 or j > 0:
        if loc and H[i][j] == 0: break
        t = T[i][j]; path.append((i, j))
        if t == 1: ra.append(a[i-1]); rb.append(b[j-1]); rm.append('|' if a[i-1] == b[j-1] else 'x'); i -= 1; j -= 1
        elif t == 2: ra.append(a[i-1]); rb.append('-'); rm.append(' '); i -= 1
        elif t == 3: ra.append('-'); rb.append(b[j-1]); rm.append(' '); j -= 1
        else: break
    ra, rb, rm = ''.join(ra[::-1]), ''.join(rb[::-1]), ''.join(rm[::-1])
    L = max(1, len(rm))
    return dict(score=best, a=ra, b=rb, mid=rm, H=np.array(H), path=path[::-1], end=(bi, bj), start=(i, j),
                identity=rm.count('|') / L, gaps=ra.count('-') + rb.count('-'), length=len(rm))

def score_only(a, b, match=2, mismatch=-1, gap=-2):
    """Two-row local alignment score: O(mn) time, O(n) memory."""
    prev = [0] * (len(b) + 1); best = 0
    for ai in a:
        cur = [0]; c0 = 0
        for j, bj in enumerate(b, 1):
            v = max(0, prev[j-1] + (match if ai == bj else mismatch), prev[j] + gap, c0 + gap)
            cur.append(v); c0 = v
            if v > best: best = v
        prev = cur
    return best


In [ ]:
%%writefile engine/workflow.py
"""Dependency-aware workflow scheduler and concurrent execution engine.

Graph + Kahn topological sort + priority queue decide WHAT may run; a thread or process pool
decides WHERE. A resource budget caps how much heavy work is in flight at once."""
import heapq, itertools, multiprocessing as mp, os, threading, time
from collections import Counter, defaultdict
from concurrent.futures import FIRST_COMPLETED, ProcessPoolExecutor, ThreadPoolExecutor, wait
from dataclasses import dataclass, field
from .align import score_only

class CycleError(Exception):
    def __init__(self, nodes): super().__init__("Dependency cycle among: " + ", ".join(sorted(nodes))); self.nodes = nodes

@dataclass
class Task:
    id: str; kind: str; payload: dict = field(default_factory=dict)
    deps: list = field(default_factory=list); priority: int = 0; cost: int = 1

# ---- task bodies: top-level functions so they can be pickled for process pools
def _gc(p):
    s, n = p['seq'], 0
    for _ in range(p.get('reps', 1)): n = sum(1 for c in s if c in 'GC')
    return dict(score=round(n / max(1, len(s)), 4), position=len(s))
def _kmer(p):
    s, k = p['seq'], p.get('k', 8)
    for _ in range(p.get('reps', 1)): c = Counter(s[i:i+k] for i in range(len(s) - k + 1))
    top = c.most_common(1)[0] if c else ('', 0)
    return dict(score=len(c), position=0, top=top[0], top_count=top[1])
def _align(p):
    return dict(score=score_only(p['a'], p['b']), position=0)
def _motif(p):
    s, m, hits = p['seq'], p['motif'], 0
    for _ in range(p.get('reps', 1)):
        hits, i = 0, s.find(m)
        while i != -1 and m: hits += 1; i = s.find(m, i + 1)
    return dict(hits=hits, position=s.find(m))
def _report(p): return dict(score=0, position=0)
RUNNERS = dict(gc=_gc, kmer=_kmer, align=_align, motif=_motif, report=_report)

def _run(kind, payload):
    t0, c0 = time.time(), time.thread_time(); res = RUNNERS[kind](payload)
    return dict(result=res, start=t0, end=time.time(), cpu=time.thread_time() - c0,
                worker=os.getpid() if mp.current_process().name != 'MainProcess' else threading.get_ident())

def toposort(tasks):
    """Kahn's algorithm with a priority queue. Raises CycleError listing the tasks that can never start."""
    ids = {t.id for t in tasks}
    for t in tasks:
        for d in t.deps:
            if d not in ids: raise KeyError(f"{t.id} depends on unknown task {d}")
    indeg = {t.id: len(set(t.deps)) for t in tasks}; kids = defaultdict(list); by = {t.id: t for t in tasks}
    for t in tasks:
        for d in set(t.deps): kids[d].append(t.id)
    cnt = itertools.count(); h = [(-t.priority, next(cnt), t.id) for t in tasks if indeg[t.id] == 0]; heapq.heapify(h)
    order, level = [], {}
    while h:
        _, _, u = heapq.heappop(h); order.append(u); level[u] = max((level[d] + 1 for d in by[u].deps), default=0)
        for v in kids[u]:
            indeg[v] -= 1
            if indeg[v] == 0: heapq.heappush(h, (-by[v].priority, next(cnt), v))
    if len(order) != len(tasks): raise CycleError(ids - set(order))
    return order, level

def critical_path(runs):
    """Longest chain of CPU time through the dependency graph: a lower bound on wall time at any worker count."""
    by = {r['id']: r for r in runs}; dur = {r['id']: r['cpu_ms'] for r in runs}; best, prev = {}, {}
    for r in sorted(runs, key=lambda r: r['end']):
        ds = [d for d in r['deps'] if d in best]
        p = max(ds, key=lambda d: best[d], default=None); best[r['id']] = dur[r['id']] + (best[p] if p else 0); prev[r['id']] = p
    if not best: return [], 0.0
    end = max(best, key=best.get); path, cur = [], end
    while cur: path.append(cur); cur = prev[cur]
    return path[::-1], best[end]

def run_workflow(tasks, workers=4, mode='threads', capacity=None):
    order, level = toposort(tasks)
    by = {t.id: t for t in tasks}; indeg = {t.id: len(set(t.deps)) for t in tasks}; kids = defaultdict(list)
    for t in tasks:
        for d in set(t.deps): kids[d].append(t.id)
    cap = max(capacity or workers, max((t.cost for t in tasks), default=1))
    if mode == 'processes':
        ctx = mp.get_context('fork' if 'fork' in mp.get_all_start_methods() else 'spawn')
        ex = ProcessPoolExecutor(max_workers=workers, mp_context=ctx)
    else: ex = ThreadPoolExecutor(max_workers=workers)
    cnt = itertools.count(); heap = []
    def push(t): heapq.heappush(heap, (-t.priority, next(cnt), t.id))
    for t in tasks:
        if indeg[t.id] == 0: push(t)
    running, used, lanes, runs, peak = {}, 0, {}, [], 0
    t0 = time.time()
    try:
        while heap or running:
            deferred = []
            while heap and len(running) < workers:
                item = heapq.heappop(heap); t = by[item[2]]
                if used + t.cost > cap and running: deferred.append(item); continue
                running[ex.submit(_run, t.kind, t.payload)] = t; used += t.cost; peak = max(peak, used)
            for d in deferred: heapq.heappush(heap, d)
            done, _ = wait(running, return_when=FIRST_COMPLETED)
            for f in done:
                t = running.pop(f); used -= t.cost; r = f.result()
                lane = lanes.setdefault(r['worker'], len(lanes))
                runs.append(dict(id=t.id, kind=t.kind, lane=lane, start=(r['start'] - t0) * 1000, end=(r['end'] - t0) * 1000,
                                 cpu_ms=r['cpu'] * 1000, result=r['result'], deps=list(t.deps), priority=t.priority, cost=t.cost))
                for c in kids[t.id]:
                    indeg[c] -= 1
                    if indeg[c] == 0: push(by[c])
    finally: ex.shutdown(wait=True)
    make = max((r['end'] for r in runs), default=0.0); serial = sum(r['cpu_ms'] for r in runs)
    path, plen = critical_path(runs)
    return dict(runs=runs, makespan=make, serial=serial, speedup=serial / make if make else 1.0,
                utilization=sum(r['cpu_ms'] for r in runs) / (make * workers) if make else 0.0,
                critical=path, critical_ms=plen, lanes=len(lanes), peak_cost=peak, levels=level, order=order)

def build_pipeline(seqs, k=8, motif='GATTACA', reps=100, align_len=300, ref=None):
    """Per sequence: load (GC) -> k-mer spectrum, alignment to the reference, motif scan; then one report."""
    names = list(seqs); ref = ref or names[0]; tasks, leaves = [], []
    for n in names:
        s = seqs[n]
        tasks.append(Task(f"load:{n}", 'gc', dict(seq=s, reps=reps), [], priority=1))
        tasks.append(Task(f"kmer:{n}", 'kmer', dict(seq=s, k=k, reps=max(1, reps // 20)), [f"load:{n}"]))
        tasks.append(Task(f"align:{n}", 'align', dict(a=seqs[ref][:align_len], b=s[:align_len]), [f"load:{n}"], priority=len(s) // 100, cost=2))
        tasks.append(Task(f"motif:{n}", 'motif', dict(seq=s, motif=motif, reps=reps), [f"load:{n}"]))
        leaves += [f"kmer:{n}", f"align:{n}", f"motif:{n}"]
    tasks.append(Task("report", 'report', {}, leaves, priority=0))
    return tasks

def counter_demo(threads=8, increments=1500, use_lock=False):
    """Read-modify-write on a shared counter. Without a lock, updates are lost; with one, none are."""
    state, lock = {'n': 0}, threading.Lock()
    def work():
        for _ in range(increments):
            if use_lock:
                with lock: v = state['n']; time.sleep(0); state['n'] = v + 1
            else: v = state['n']; time.sleep(0); state['n'] = v + 1
    ts = [threading.Thread(target=work) for _ in range(threads)]; t = time.perf_counter()
    for x in ts: x.start()
    for x in ts: x.join()
    return state['n'], threads * increments, time.perf_counter() - t


In [ ]:
%%writefile engine/store.py
"""PostgreSQL-style relational schema on SQLite, so the workbench runs anywhere with no server."""
import sqlite3, threading, time, random
import pandas as pd

SCHEMA = """
CREATE TABLE samples(sample_id INTEGER PRIMARY KEY, name TEXT UNIQUE NOT NULL);
CREATE TABLE sequences(seq_id INTEGER PRIMARY KEY, sample_id INTEGER NOT NULL REFERENCES samples(sample_id),
  name TEXT NOT NULL, kind TEXT NOT NULL CHECK(kind IN ('DNA','PROTEIN')),
  length INTEGER NOT NULL CHECK(length > 0), gc REAL, residues TEXT NOT NULL);
CREATE TABLE jobs(job_id INTEGER PRIMARY KEY, run_id INTEGER NOT NULL, task TEXT NOT NULL,
  seq_id INTEGER REFERENCES sequences(seq_id), priority INTEGER NOT NULL DEFAULT 0,
  status TEXT NOT NULL DEFAULT 'pending' CHECK(status IN ('pending','running','done','failed')));
CREATE TABLE executions(exec_id INTEGER PRIMARY KEY, job_id INTEGER NOT NULL REFERENCES jobs(job_id),
  worker INTEGER, started_ms REAL, finished_ms REAL, cpu_ms REAL);
CREATE TABLE results(result_id INTEGER PRIMARY KEY, job_id INTEGER NOT NULL REFERENCES jobs(job_id),
  seq_id INTEGER REFERENCES sequences(seq_id), position INTEGER, score REAL, detail TEXT);
"""
INDEXES = {
    'idx_seq_sample': "CREATE INDEX idx_seq_sample ON sequences(sample_id)",
    'idx_seq_length': "CREATE INDEX idx_seq_length ON sequences(length)",
    'idx_jobs_run': "CREATE INDEX idx_jobs_run ON jobs(run_id)",
    'idx_exec_job': "CREATE INDEX idx_exec_job ON executions(job_id)",
    'idx_res_seq': "CREATE INDEX idx_res_seq ON results(seq_id)"}

class Store:
    def __init__(self):
        self.lock = threading.Lock(); self._open()
    def _open(self):
        self.con = sqlite3.connect(':memory:', check_same_thread=False)
        self.con.execute("PRAGMA foreign_keys=ON"); self.con.executescript(SCHEMA); self.con.commit()
        self.run_id = 0; self.indexed = False
    def reset(self):
        with self.lock: self.con.close(); self._open()
    def add_sequences(self, sample, seqs, kind_of, gc):
        with self.lock, self.con:
            cur = self.con.execute("INSERT INTO samples(name) VALUES(?)", (sample,)); sid = cur.lastrowid
            self.con.executemany("INSERT INTO sequences(sample_id,name,kind,length,gc,residues) VALUES(?,?,?,?,?,?)",
                                 [(sid, n, kind_of(s), len(s), gc(s), s) for n, s in seqs.items()])
    def seq_ids(self):
        with self.lock: return dict(self.con.execute("SELECT name, seq_id FROM sequences"))
    def df(self, sql, params=()):
        with self.lock: return pd.read_sql_query(sql, self.con, params=params)
    def explain(self, sql):
        with self.lock: return [r[3] for r in self.con.execute("EXPLAIN QUERY PLAN " + sql)]
    def timed(self, sql, reps=20):
        with self.lock:
            best = 1e9
            for _ in range(reps):
                t = time.perf_counter(); self.con.execute(sql).fetchall(); best = min(best, time.perf_counter() - t)
        return best * 1000
    def set_indexes(self, on):
        with self.lock, self.con:
            for n, ddl in INDEXES.items():
                self.con.execute(f"DROP INDEX IF EXISTS {n}")
                if on: self.con.execute(ddl)
            self.indexed = on
    def record_run(self, runs, seq_of):
        """Persist one workflow run in a single transaction: jobs, their executions and their results."""
        with self.lock, self.con:
            self.run_id += 1; ids = {}
            for r in runs:
                sid = seq_of.get(r['id'].split(':', 1)[-1])
                c = self.con.execute("INSERT INTO jobs(run_id,task,seq_id,priority,status) VALUES(?,?,?,?,'done')", (self.run_id, r['id'], sid, r['priority']))
                ids[r['id']] = (c.lastrowid, sid)
                self.con.execute("INSERT INTO executions(job_id,worker,started_ms,finished_ms,cpu_ms) VALUES(?,?,?,?,?)", (c.lastrowid, r['lane'], r['start'], r['end'], r['cpu_ms']))
                res = r['result'] or {}
                if 'score' in res or 'hits' in res:
                    self.con.execute("INSERT INTO results(job_id,seq_id,position,score,detail) VALUES(?,?,?,?,?)",
                                     (c.lastrowid, sid, res.get('position'), res.get('score', res.get('hits')), r['kind']))
            return self.run_id
    def index_experiment(self, n, reps=15):
        """Time a filter and a join on n synthetic rows with and without indexes."""
        rnd = random.Random(5)
        with self.lock, self.con:
            c = self.con
            for t in ('bench_rows', 'bench_samples'): c.execute(f"DROP TABLE IF EXISTS {t}")
            c.execute("CREATE TABLE bench_samples(sample_id INTEGER PRIMARY KEY, name TEXT)")
            c.execute("CREATE TABLE bench_rows(seq_id INTEGER PRIMARY KEY, sample_id INT REFERENCES bench_samples, length INT, gc REAL)")
            c.executemany("INSERT INTO bench_samples VALUES(?,?)", [(i, f"s{i}") for i in range(1000)])
            c.executemany("INSERT INTO bench_rows VALUES(?,?,?,?)", ((i, rnd.randrange(1000), rnd.randrange(200, 5000), rnd.random()) for i in range(n)))
        qs = {'Filter by sample': "SELECT COUNT(*), AVG(gc) FROM bench_rows WHERE sample_id=7",
              'Join with filter': "SELECT s.name, COUNT(*) FROM bench_rows q JOIN bench_samples s USING(sample_id) WHERE q.length>4900 GROUP BY s.name"}
        rows = []
        for ix in (False, True):
            with self.lock, self.con:
                self.con.execute("DROP INDEX IF EXISTS b1"); self.con.execute("DROP INDEX IF EXISTS b2")
                if ix: self.con.execute("CREATE INDEX b1 ON bench_rows(sample_id)"); self.con.execute("CREATE INDEX b2 ON bench_rows(length)")
            for name, q in qs.items(): rows.append(dict(query=name, indexed='With indexes' if ix else 'No indexes', ms=self.timed(q, reps)))
        return pd.DataFrame(rows)


In [ ]:
%%writefile engine/bench.py
"""Live benchmarks that run on the machine hosting the app."""
import random, time, tracemalloc
import pandas as pd
from .seqtools import KmerIndex, naive_find
from .align import align, score_only

def _best(f, reps):
    b = 1e9
    for _ in range(reps):
        t = time.perf_counter(); f(); b = min(b, time.perf_counter() - t)
    return b * 1000

def bench_search(sizes, qlen=24, k=11, reps=3, seed=1):
    rng = random.Random(seed); rows = []
    for n in sizes:
        s = ''.join(rng.choices('ACGT', k=n)); p = rng.randrange(n - qlen); q = s[p:p+qlen]
        def loop(): return [i for i in range(n - qlen + 1) if s[i:i+qlen] == q]
        tracemalloc.start(); t = time.perf_counter(); idx = KmerIndex(k); idx.add('s', s); build = (time.perf_counter() - t) * 1000
        peak = tracemalloc.get_traced_memory()[1] / 1e6; tracemalloc.stop()
        assert [x for _, x in idx.exact(q)] == loop() == naive_find(s, q)
        t = time.perf_counter()
        for _ in range(500): idx.exact(q)
        look = (time.perf_counter() - t) / 500 * 1000
        rows += [dict(n=n, method='Python loop scan', ms=_best(loop, reps), mb=None),
                 dict(n=n, method='str.find scan (C)', ms=_best(lambda: naive_find(s, q), reps), mb=None),
                 dict(n=n, method='K-mer index build (once)', ms=build, mb=peak),
                 dict(n=n, method='K-mer index lookup', ms=look, mb=None)]
    return pd.DataFrame(rows)

def bench_align(lengths, reps=1, seed=2):
    rng = random.Random(seed); rows = []
    for L in lengths:
        a, b = ''.join(rng.choices('ACGT', k=L)), ''.join(rng.choices('ACGT', k=L))
        rows.append(dict(length=L, method='Two-row score', ms=_best(lambda: score_only(a, b), reps)))
        if L <= 800: rows.append(dict(length=L, method='Full matrix + traceback', ms=_best(lambda: align(a, b), reps)))
    return pd.DataFrame(rows)


## 0 · Setup
The `engine` package holds the system: sequence tools and k-mer index (`seqtools`), alignment (`align`), the workflow scheduler and execution engine (`workflow`), the relational store (`store`) and benchmarks (`bench`). This notebook walks through each part with your own data.

In [ ]:
import os, sys, time, random
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from matplotlib.patches import Patch
from engine import seqtools as sq, align as al, workflow as wf, bench as bn
from engine.store import Store

BLUE, ORANGE, GREEN, SLATE, PLUM = "#0b63b5", "#e07a1f", "#1c8c6e", "#5b6b7c", "#8a4fb0"
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.alpha": .25})
print("CPUs on this machine:", os.cpu_count())

## 1 · Data
Generate synthetic sequences (some related to sequence 1, with a planted motif) and store them in the relational schema. To use your own data, see section 8.

In [ ]:
seqs = sq.synthetic(n=12, length=2000, gc_target=.5, motif="GATTACAGATTACA", related=.4, seed=1)
store = Store()
store.add_sequences("synthetic", seqs, sq.kind_of, sq.gc)
df = store.df("SELECT name, kind, length, ROUND(gc, 3) AS gc FROM sequences ORDER BY seq_id")
display(df)
fig, ax = plt.subplots(1, 2, figsize=(9, 2.8))
ax[0].hist(df.length, bins=10, color=BLUE); ax[0].set(title="Length distribution", xlabel="Length", ylabel="Sequences")
ax[1].scatter(df.length, df.gc, color=ORANGE); ax[1].set(title="GC against length", xlabel="Length", ylabel="GC")
plt.tight_layout(); plt.show()

## 2 · Search
### 2.1 K-mer index against a full scan
The index maps every k-mer to the places it occurs. A query becomes one lookup plus verification. We check that it returns exactly what a scan returns.

In [ ]:
k = 8
idx, build_s, peak_mb = sq.build_index(seqs, k)
print(idx.stats(), f"| build {build_s*1000:.1f} ms, peak {peak_mb:.1f} MB")

query = "GATTACAGATTACA"
t = time.perf_counter(); hits = idx.exact(query); t_index = (time.perf_counter() - t) * 1000
t = time.perf_counter(); ref = [(n, p) for n, s in seqs.items() for p in sq.naive_find(s, query)]; t_scan = (time.perf_counter() - t) * 1000
assert sorted(hits) == sorted(ref), "index and scan disagree"
print(f"{len(hits)} matches | index {t_index:.4f} ms | scan {t_scan:.4f} ms | scan is {t_scan/max(t_index,1e-9):.0f}x slower")

names = list(seqs); fig, ax = plt.subplots(figsize=(9, 0.35 * len(names) + 1))
for i, n in enumerate(names): ax.hlines(i, 0, len(seqs[n]), color="#c9d8e4", lw=6)
ax.scatter([p for _, p in hits], [names.index(n) for n, _ in hits], marker="|", s=250, color=ORANGE, zorder=3)
ax.set_yticks(range(len(names)), names); ax.invert_yaxis(); ax.set(title="Where the query matches", xlabel="Position"); plt.show()

### 2.2 Search with mismatches (seed and vote)
Every query k-mer votes for a start position; candidates are verified by Hamming distance. A guarantee of finding every match holds when the number of k-mers exceeds `d × k`.

In [ ]:
true_seq, true_pos = "seq_001", 100
q = seqs[true_seq][true_pos:true_pos + 30]
mq = sq.mutate(q, .06, random.Random(1), indel=0); d = sq.hamming(q, mq)
rows = idx.approx(mq, d)
print(f"query has {d} mismatches; guaranteed to find all matches: {idx.guaranteed(len(mq), d)}")
display(pd.DataFrame(rows, columns=["sequence", "position", "mismatches", "seed votes"]).head(10))

### 2.3 Trie prefix search
A trie stores shared prefixes once. The chart shows where it branches fully and where it splits into single chains.

In [ ]:
trie = sq.Trie()
for n, s in seqs.items(): trie.insert(s[:24], n)
prefix = seqs["seq_001"][:8]
print("sequences starting with", prefix, "->", trie.prefix(prefix), "| trie nodes:", trie.nodes)
level, per = [trie.root], []
for depth in range(1, 25):
    level = [v for nd in level for kk, v in nd.items() if kk != "$"]; per.append(len(level))
plt.figure(figsize=(6, 2.6)); plt.plot(range(1, 25), per, marker="o", color=BLUE); plt.title("Trie nodes per depth"); plt.xlabel("Depth"); plt.ylabel("Nodes"); plt.show()

## 3 · Alignment
Smith-Waterman (local) fills a scoring matrix and traces back from the best cell. The orange path is the alignment.

In [ ]:
A, B = seqs["seq_001"][:200], seqs["seq_002"][:200]
r = al.align(A, B, mode="local", match=2, mismatch=-1, gap=-2)
print(f"score {r['score']} | identity {r['identity']*100:.0f}% | gaps {r['gaps']} | length {r['length']}")
print("\n".join(s[:100] for s in (r["a"], r["mid"], r["b"])))
fig, ax = plt.subplots(1, 2, figsize=(11, 4.4))
ax[0].imshow(r["H"], cmap="Blues", aspect="auto"); ax[0].plot([p[1] for p in r["path"]], [p[0] for p in r["path"]], color=ORANGE, lw=2)
ax[0].set(title="Scoring matrix with traceback", xlabel="Position in B", ylabel="Position in A"); ax[0].grid(False)
w = 9; tb = {}
for j in range(len(B) - w + 1): tb.setdefault(B[j:j+w], []).append(j)
pts = [(i, j) for i in range(len(A) - w + 1) for j in tb.get(A[i:i+w], ())]
ax[1].scatter([j for _, j in pts], [i for i, _ in pts], s=6, color=BLUE); ax[1].invert_yaxis(); ax[1].set(title=f"Dot plot (word size {w})", xlabel="B", ylabel="A")
plt.tight_layout(); plt.show()

## 4 · Workflow engine
Each sequence becomes a small graph of tasks (load, k-mer spectrum, alignment, motif scan) that feeds one report. Kahn's algorithm orders the graph and detects cycles; a priority queue releases ready tasks; a resource budget caps heavy work in flight; a worker pool executes them for real.

In [ ]:
tasks = wf.build_pipeline(seqs, k=8, motif="GATTACA", reps=100, align_len=300)
order, level = wf.toposort(tasks)
print(f"{len(tasks)} tasks in {max(level.values()) + 1} dependency levels; first five in order: {order[:5]}")

try:   # a cycle is refused, not run
    wf.toposort([wf.Task("a", "gc", deps=["b"]), wf.Task("b", "gc", deps=["a"])])
except wf.CycleError as e:
    print("cycle detected:", e)

In [ ]:
COLORS = dict(gc=BLUE, kmer=GREEN, align=ORANGE, motif=PLUM, report=SLATE)

def gantt(res, title):
    fig, ax = plt.subplots(figsize=(10, 1.3 + 0.5 * res["lanes"]))
    for r in res["runs"]:
        crit = r["id"] in res["critical"]
        ax.barh(r["lane"], r["end"] - r["start"], left=r["start"], color=COLORS[r["kind"]], edgecolor="black" if crit else "white", linewidth=1.8 if crit else .5)
    ax.set_yticks(range(res["lanes"]), [f"worker {i+1}" for i in range(res["lanes"])]); ax.invert_yaxis()
    ax.legend(handles=[Patch(color=c, label=k) for k, c in COLORS.items()], ncol=5, loc="upper center", bbox_to_anchor=(.5, -.25), frameon=False)
    ax.set(title=title, xlabel="Milliseconds since start"); plt.show()

workers, mode = 4, "threads"          # try "processes" too
res = wf.run_workflow(tasks, workers=workers, mode=mode, capacity=workers + 2)
print(f"wall {res['makespan']:.0f} ms | CPU work {res['serial']:.0f} ms | speedup {res['speedup']:.2f}x | workers used {res['lanes']} | peak budget {res['peak_cost']}")
print(f"critical path (black outline): {' -> '.join(res['critical'][:5])} ... ({res['critical_ms']:.0f} ms of CPU time)")
gantt(res, f"Real execution on {res['lanes']} {mode[:-1]} worker(s)")

In [ ]:
rows = []
for w in (1, 2, 4, 8):
    rr = wf.run_workflow(tasks, workers=w, mode="threads", capacity=w + 2); rows.append((w, rr["makespan"]))
sweep = pd.DataFrame(rows, columns=["workers", "wall_ms"])
plt.figure(figsize=(6, 3)); plt.plot(sweep.workers, sweep.wall_ms, marker="o", color=BLUE, label="Measured")
plt.plot(sweep.workers, sweep.wall_ms.iloc[0] / sweep.workers, "--", color=SLATE, label="Perfect scaling")
plt.xlabel("Workers"); plt.ylabel("Wall time (ms)"); plt.legend(); plt.title(f"Scaling on this machine ({os.cpu_count()} CPU)"); plt.show()
print("Pure-Python tasks hold the global interpreter lock, so threads rarely scale. Re-run with mode='processes' on a multi-core machine.")

### Synchronization: why shared state needs a lock
Each thread reads a counter, yields, then writes back read+1. Without mutual exclusion, updates are lost.

In [ ]:
no_lock, expected, t1 = wf.counter_demo(8, 1000, use_lock=False)
locked, _, t2 = wf.counter_demo(8, 1000, use_lock=True)
print(f"expected {expected:,} | without lock {no_lock:,} ({expected-no_lock:,} lost updates) | with lock {locked:,} ({t2/t1:.1f}x the time)")

## 5 · Relational store
Every workflow run is saved: jobs, their executions and their results, in one transaction. The query plan shows what the database does before and after indexing.

In [ ]:
run_id = store.record_run(res["runs"], store.seq_ids())
display(store.df("""
SELECT substr(j.task, 1, instr(j.task, ':') - 1) AS kind, COUNT(*) AS jobs, ROUND(SUM(e.finished_ms - e.started_ms)) AS total_ms
FROM jobs j JOIN executions e USING(job_id) WHERE instr(j.task, ':') > 0 GROUP BY kind ORDER BY total_ms DESC"""))
sql = "SELECT * FROM sequences WHERE sample_id = 1"
print("without index:", store.explain(sql)); store.set_indexes(True); print("with index:   ", store.explain(sql))

In [ ]:
exp = store.index_experiment(100_000)
pv = exp.pivot(index="query", columns="indexed", values="ms")[["No indexes", "With indexes"]]
ax = pv.plot.bar(logy=True, color=[SLATE, BLUE], figsize=(6, 3), rot=0); ax.set(ylabel="ms per query (log)", title="Indexes at 100,000 rows", xlabel=""); plt.show()
print((pv["No indexes"] / pv["With indexes"]).round(0).astype(int).astype(str) + "x faster with indexes")

## 6 · Benchmarks on this machine
Search and alignment cost against size, measured here. Each search run also asserts that the scan and the index return identical matches.

In [ ]:
s = bn.bench_search([10_000, 30_000, 100_000, 300_000], qlen=24, k=11, reps=3)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
for m, g in s.groupby("method"): ax[0].plot(g.n, g.ms, marker="o", label=m)
ax[0].set(xscale="log", yscale="log", title="Search cost against reference size", xlabel="Bases", ylabel="ms"); ax[0].legend(fontsize=8)
a = bn.bench_align([50, 100, 200, 400])
for m, g in a.groupby("method"): ax[1].plot(g.length, g.ms, marker="o", label=m)
ax[1].set(title="Alignment time against length", xlabel="Length (both sequences)", ylabel="ms"); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.show()
display(s.pivot(index="n", columns="method", values="ms").round(4))

## 7 · What the numbers mean
Scans grow with the reference while a lookup stays flat, so the index repays its build after a handful of queries. Alignment cost is quadratic. Workflow speedup is bounded by the critical path and, in pure Python threads, by the interpreter lock. Absolute times depend on this machine; the shapes are the result.

## 8 · Use your own data
Upload a FASTA file (or paste text), then re-run the sections above with `seqs` replaced.

In [ ]:
from google.colab import files
up = files.upload()                       # choose a .fasta file
seqs = sq.parse_fasta(next(iter(up.values())).decode()); print({n: len(s) for n, s in seqs.items()})

## 9 · Real example data
Bundled public-domain sequences live in `app/data` (catalogued in `datasets.json`). Load one and re-run the search/alignment sections above with `seqs` replaced.


In [ ]:
import pathlib
demo = None
for p in (pathlib.Path("../app/data/NC_000932_demo10k.fasta"), pathlib.Path("app/data/NC_000932_demo10k.fasta")):
    if p.exists(): demo = p.read_text(); break
if demo is None:  # running on Colab without the repo: fetch the bundled 10 kb demo from GitHub
    try:
        import urllib.request
        demo = urllib.request.urlopen("https://raw.githubusercontent.com/YOUR_GITHUB_USER/YOUR_REPO/main/app/data/NC_000932_demo10k.fasta", timeout=20).read().decode()
    except Exception:
        print("Bundled FASTA not reachable in this runtime (run set_links.py so the URL above is real); using synthetic data instead.")
        seqs = sq.synthetic(n=4, length=2500, seed=7)
if demo is not None:
    seqs = sq.parse_fasta(demo); print({n: len(s) for n, s in seqs.items()})


---
**Links:** [Website](YOUR_SITE_URL) · [Streamlit workbench](YOUR_STREAMLIT_URL) · [Source on GitHub](https://github.com/YOUR_GITHUB_USER/YOUR_REPO)
**Contact:** harshinisuresha7@gmail.com